**AWS Athena**

* Directly query S3
* S3 (Bronze) → Athena → SQL → OBT

**Create Database**

In [ ]:
CREATE DATABASE uber;

**Create Bronze Table**

This reads your JSON files from S3

In [ ]:
CREATE EXTERNAL TABLE uber.bronze_rides (
    ride_id STRING,
    confirmation_number STRING,
    passenger_id STRING,
    driver_id STRING,
    vehicle_id STRING,
    pickup_location_id STRING,
    dropoff_location_id STRING,
    vehicle_type_id INT,
    vehicle_make_id INT,
    payment_method_id INT,
    ride_status_id INT,
    pickup_city_id INT,
    dropoff_city_id INT,
    cancellation_reason_id INT,
    passenger_name STRING,
    passenger_email STRING,
    passenger_phone STRING,
    driver_name STRING,
    driver_rating DOUBLE,
    driver_phone STRING,
    driver_license STRING,
    vehicle_model STRING,
    vehicle_color STRING,
    license_plate STRING,
    pickup_address STRING,
    pickup_latitude DOUBLE,
    pickup_longitude DOUBLE,
    dropoff_address STRING,
    dropoff_latitude DOUBLE,
    dropoff_longitude DOUBLE,
    distance_miles DOUBLE,
    duration_minutes INT,
    booking_timestamp STRING,
    pickup_timestamp STRING,
    dropoff_timestamp STRING,
    base_fare DOUBLE,
    distance_fare DOUBLE,
    time_fare DOUBLE,
    surge_multiplier DOUBLE,
    subtotal DOUBLE,
    tip_amount DOUBLE,
    total_fare DOUBLE,
    rating DOUBLE
)
ROW FORMAT SERDE 'org.openx.data.jsonserde.JsonSerDe'
LOCATION 's3://uber-data-lake-pooja/bronze/streaming/';

**Test Query**

In [ ]:
SELECT * FROM uber.bronze_rides LIMIT 10;

**Create Silver (OBT)**

In [ ]:
CREATE TABLE uber.silver_obt AS
SELECT
    *,
    CASE 
        WHEN total_fare > 100 THEN 'High'
        WHEN total_fare > 50 THEN 'Medium'
        ELSE 'Low'
    END AS ride_value_category
FROM uber.bronze_rides;

**Fix Timestamps, Remove Duplicates, Add Derived Columns**

In [ ]:
CREATE VIEW uber.silver_obt AS
SELECT DISTINCT
    ride_id,
    passenger_id,
    driver_id,
    vehicle_id,

    date_parse(replace(substr(booking_timestamp, 1, 19), 'T', ' '), '%Y-%m-%d %H:%i:%s') AS booking_time,
    date_parse(replace(substr(pickup_timestamp, 1, 19), 'T', ' '), '%Y-%m-%d %H:%i:%s') AS pickup_time,
    date_parse(replace(substr(dropoff_timestamp, 1, 19), 'T', ' '), '%Y-%m-%d %H:%i:%s') AS dropoff_time,

    distance_miles,
    duration_minutes,
    total_fare,
    tip_amount,
    surge_multiplier,

    (total_fare - tip_amount) AS net_fare,

    CASE 
        WHEN total_fare > 100 THEN 'High'
        WHEN total_fare > 50 THEN 'Medium'
        ELSE 'Low'
    END AS ride_value_category,

    CASE
        WHEN duration_minutes < 15 THEN 'Short'
        WHEN duration_minutes < 45 THEN 'Medium'
        ELSE 'Long'
    END AS trip_duration_category,

    CASE
        WHEN surge_multiplier > 1.5 THEN 'Peak'
        ELSE 'Normal'
    END AS surge_category,

    vehicle_type_id,
    vehicle_make_id,
    payment_method_id,
    ride_status_id,
    pickup_city_id,
    dropoff_city_id,
    cancellation_reason_id

FROM uber.bronze_rides;

**Verify**

In [ ]:
SELECT * FROM uber.silver_obt LIMIT 10;

**FastAPI → Kinesis → Lambda → S3 → Athena (Bronze + Clean Silver)**

**Gold Layer (Fact + Dimension Tables)**

**Fact table → rides**

In [ ]:
CREATE OR REPLACE VIEW uber.fact_rides AS
SELECT
    f.*,
    l.city,
    l.region,
    v.vehicle_type,
    
    -- business metrics
    (f.total_fare / f.distance_miles) AS fare_per_mile,
    (f.total_fare / f.duration_minutes) AS fare_per_minute

FROM uber.fact_rides f
LEFT JOIN uber.dim_location_enriched l
    ON f.pickup_city_id = l.pickup_city_id
LEFT JOIN uber.dim_vehicle_enriched v
    ON f.vehicle_id = v.vehicle_id;

Dimension Tables

Passenger Dimension

In [ ]:
CREATE TABLE uber.dim_passenger AS
SELECT DISTINCT
    passenger_id
FROM uber.silver_obt;

Driver Dimension

In [ ]:
CREATE TABLE uber.dim_driver AS
SELECT DISTINCT
    driver_id
FROM uber.silver_obt;

Vehicle Dimension

In [ ]:
CREATE OR REPLACE VIEW uber.dim_vehicle AS
SELECT DISTINCT
    s.vehicle_id,
    s.vehicle_type_id,
    v.vehicle_type,
    v.base_rate,
    v.per_mile,
    v.per_minute
FROM uber.silver_obt s
LEFT JOIN uber.map_vehicle_types v
ON s.vehicle_type_id = v.vehicle_type_id;

Payment Dimension

In [ ]:
CREATE TABLE uber.dim_payment AS
SELECT DISTINCT
    payment_method_id
FROM uber.silver_obt;

Location Dimension

In [ ]:
CREATE OR REPLACE VIEW uber.dim_location AS
SELECT DISTINCT
    s.pickup_city_id,
    c.city,
    c.state,
    c.region
FROM uber.silver_obt s
LEFT JOIN uber.map_cities c
ON s.pickup_city_id = c.city_id;

Booking Dimension

In [ ]:
CREATE TABLE uber.dim_booking AS
SELECT DISTINCT
    ride_id,
    ride_status_id,
    cancellation_reason_id
FROM uber.silver_obt;

dim_passenger
dim_driver
dim_vehicle
dim_payment
dim_location
dim_booking
    ↓
FACT TABLE

Create Mapping Tables

City Mapping

In [ ]:
CREATE EXTERNAL TABLE uber.map_cities (
    city_id INT,
    city STRING,
    state STRING,
    region STRING
)
ROW FORMAT SERDE 'org.openx.data.jsonserde.JsonSerDe'
LOCATION 's3://uber-data-lake-pooja/bronze/reference/map_cities/';

Vehicle Type Mapping

In [ ]:
CREATE EXTERNAL TABLE uber.map_vehicle_types (
    vehicle_type_id INT,
    vehicle_type STRING,
    description STRING,
    base_rate DOUBLE,
    per_mile DOUBLE,
    per_minute DOUBLE
)
ROW FORMAT SERDE 'org.openx.data.jsonserde.JsonSerDe'
LOCATION 's3://uber-data-lake-pooja/bronze/reference/map_vehicle_types/';

Data Quality Checks

In [ ]:
-- Null check
SELECT COUNT(*) FROM uber.silver_obt WHERE ride_id IS NULL;

-- Duplicate check
SELECT ride_id, COUNT(*)
FROM uber.silver_obt
GROUP BY ride_id
HAVING COUNT(*) > 1;

Create SCD Table

In [ ]:
CREATE TABLE uber.dim_location_scd
WITH (
    format = 'PARQUET'
) AS
SELECT
    pickup_city_id AS city_id,
    CAST(NULL AS VARCHAR) AS city,
    CAST(NULL AS VARCHAR) AS state,
    CAST(NULL AS VARCHAR) AS region,
    
    CAST(current_date AS VARCHAR) AS effective_date,
    CAST(NULL AS VARCHAR) AS end_date,
    
    TRUE AS is_current

FROM uber.silver_obt
WHERE 1=0;

Initial Load

In [ ]:
CREATE TABLE uber.dim_location_scd AS
SELECT
    pickup_city_id AS city_id,
    'CITY_NAME' AS city,
    'STATE' AS state,
    'REGION' AS region,

    CAST(current_date AS VARCHAR) AS effective_date,
    NULL AS end_date,

    TRUE AS is_current

FROM uber.silver_obt;

SCD TYPE 2

Close old records

In [ ]:
UPDATE uber.dim_location_scd
SET 
    end_date = CAST(current_date AS VARCHAR),
    is_current = FALSE
WHERE city_id IN (
    SELECT pickup_city_id FROM uber.silver_obt
)
AND is_current = TRUE;

Insert new records

In [ ]:
CREATE TABLE uber.dim_location_scd AS
SELECT
    pickup_city_id AS city_id,
    'CITY_NAME' AS city,
    'STATE' AS state,
    'REGION' AS region,

    CAST(current_date AS VARCHAR) AS effective_date,
    NULL AS end_date,

    TRUE AS is_current

FROM uber.silver_obt;

In [ ]:
CREATE TABLE uber.dim_location_scd AS
SELECT DISTINCT
    s.pickup_city_id AS city_id,
    c.city,
    c.state,
    c.region,

    CAST(current_date AS VARCHAR) AS effective_date,
    CAST(NULL AS VARCHAR) AS end_date,

    TRUE AS is_current

FROM uber.silver_obt s
LEFT JOIN uber.map_cities c
ON s.pickup_city_id = c.city_id;

Query Current Records

In [ ]:
SELECT *
FROM uber.dim_location_scd
WHERE is_current = TRUE;

Query History

In [ ]:
SELECT *
FROM uber.dim_location_scd
WHERE city_id = 5;

silver_obt + map_cities → enriched SCD dimension

Create Mapping Table

In [ ]:
CREATE EXTERNAL TABLE uber.map_cities (
    city_id INT,
    city STRING,
    state STRING,
    region STRING
)
ROW FORMAT SERDE 'org.openx.data.jsonserde.JsonSerDe'
LOCATION 's3://uber-data-lake-pooja/bronze/reference/map_cities/';

Test

In [ ]:
SELECT * FROM uber.map_cities LIMIT 10;

Business Queries (Insights Layer)

Revenue by City

In [ ]:
SELECT
    d.city,
    SUM(f.total_fare) AS total_revenue
FROM uber.fact_rides f
JOIN uber.dim_location_scd d
ON f.pickup_city_id = d.city_id
WHERE d.is_current = TRUE
GROUP BY d.city
ORDER BY total_revenue DESC;

Average Fare per City

In [ ]:
SELECT
    d.city,
    AVG(f.total_fare) AS avg_fare
FROM uber.fact_rides f
JOIN uber.dim_location_scd d
ON f.pickup_city_id = d.city_id
WHERE d.is_current = TRUE
GROUP BY d.city;

Peak Surge Analysis

In [ ]:
SELECT
    surge_multiplier,
    COUNT(*) AS ride_count
FROM uber.fact_rides
GROUP BY surge_multiplier
ORDER BY surge_multiplier DESC;

Ride Distribution by Region

In [ ]:
SELECT
    d.region,
    COUNT(*) AS total_rides
FROM uber.fact_rides f
JOIN uber.dim_location_scd d
ON f.pickup_city_id = d.city_id
WHERE d.is_current = TRUE
GROUP BY d.region;